# Mercedes validation: frozen calibration, held-out report and paired videos

The calibration was developed using only the 30, 60 and 100 km/h pilot clips.
The other 27 filenames supply evaluation references only after prediction.
Text label files are ignored. No neural-network weights are fine-tuned here.
This CPU notebook consumes the completed speed-blind GPU inference notebook and the private source videos.
It writes a before/after report, per-clip errors and separate calibration/speed videos.
Cruise-control settings are reference values, not independent radar measurements.


In [ ]:
from pathlib import Path
import json,subprocess,sys
sources={'calibrate_speed.py': '"""Experimental local road rectification from learned cuboids and explicit car-size priors.\n\nIndependent implementation of rectangle/aspect/rigidity fitting, inspired by\nSochor et al. (2017) and the UrbanOmniDetect (2026) ground-footprint approach.\nRanges are sensitivity envelopes, not confidence intervals or accuracy claims.\n"""\nfrom pathlib import Path\nfrom collections import defaultdict\nimport csv\nimport json\nimport math\nimport os\nimport cv2\nimport numpy as np\nfrom scipy.optimize import least_squares\n\nROOT = Path(os.environ.get(\'FYP_SPEED_ROOT\',str(Path(__file__).resolve().parent)))\ndownloaded_geometry = ROOT / \'outputs\' / \'geometry\'\ndefault_geometry = downloaded_geometry if downloaded_geometry.exists() else ROOT / \'output\' / \'inputs\' / \'geometry\'\nGEOMETRY = Path(os.environ.get(\'FYP_GEOMETRY_DIR\', str(default_geometry)))\nOUT = Path(os.environ.get(\'FYP_SPEED_OUTPUT\',str(ROOT / \'outputs\' / \'speed_experiment\')))\nLENGTH_M, WIDTH_M = 4.5, 1.8\nDIMENSION_SCENARIOS = [(4.5,1.8),(3.2,1.45),(3.2,2.0),(5.2,1.45),(5.2,2.0)]\n\n\ndef project(points, H):\n    p = np.asarray(points, dtype=float)\n    flat = p.reshape(-1,2)\n    result = np.c_[flat,np.ones(len(flat))] @ H.T\n    denominator = result[:,2]\n    safe = np.where(np.abs(denominator)<1e-8, np.where(denominator>=0,1e-8,-1e-8),denominator)\n    return (result[:,:2]/safe[:,None]).reshape(p.shape)\n\n\ndef compose(theta, normalizer):\n    p,q,a,b = theta\n    return np.array([[np.exp(a),b,0],[0,1,0],[p,q,1]]) @ normalizer\n\n\ndef measures(quads):\n    edges = np.roll(quads,-1,axis=1)-quads\n    sizes = np.linalg.norm(edges,axis=2)+1e-9\n    unit = edges/sizes[:,:,None]\n    cosines = np.sum(unit*np.roll(unit,-1,axis=1),axis=2)\n    opposite = unit[:,:2,0]*unit[:,2:,1]-unit[:,:2,1]*unit[:,2:,0]\n    lengths=(sizes[:,0]+sizes[:,2])/2\n    widths=(sizes[:,1]+sizes[:,3])/2\n    return cosines,opposite,lengths,widths\n\n\ndef fit_mapping(records, width, height, length_m=LENGTH_M, width_m=WIDTH_M, fast=False, legacy_normalization=False):\n    quads=np.array([r[\'quad\'] for r in records],dtype=float)\n    ids=np.array([r[\'track_id\'] for r in records])\n    unique=np.unique(ids)\n    center=np.median(quads.reshape(-1,2),axis=0)\n    normalizer=np.array([[2/width,0,-2*center[0]/width],[0,2/width,-2*center[1]/width],[0,0,1]])\n    if legacy_normalization:\n        normalizer=np.array([[2/width,0,-1],[0,2/width,-height/width],[0,0,1]])\n    weights=np.array([1/math.sqrt(np.sum(ids==i)) for i in ids])\n    weights/=np.mean(weights)\n    groups=[np.flatnonzero(ids==i) for i in unique]\n\n    def residual(theta):\n        H=compose(theta,normalizer)\n        world=project(quads,H)\n        angles,parallel,lengths,widths=measures(world)\n        rigidity=np.zeros(len(records))\n        for group in groups:\n            rigidity[group]=np.log(lengths[group]/np.median(lengths[group]))\n        shape=np.log((lengths/widths)/(length_m/width_m))\n        geometric=np.c_[angles,parallel*0.5,shape*0.65,rigidity*0.5]*weights[:,None]\n        den=(np.c_[quads.reshape(-1,2),np.ones(len(quads)*4)]@H.T)[:,2]\n        pole=np.maximum(0,0.08-den)*15\n        return np.r_[geometric.ravel(),pole]\n\n    q_starts=([0,1,2] if not fast else [1,2]) if legacy_normalization else ([0,2,5,10,20] if not fast else [2,10])\n    aspect_starts=([0,0.7] if not fast else [0]) if legacy_normalization else ([-3,0] if not fast else [-2])\n    starts=[(0,q,a,0) for q in q_starts for a in aspect_starts]\n    fits=[]\n    for theta in starts:\n        bound=4 if legacy_normalization else 100\n        shape_bound=2.5 if legacy_normalization else 7\n        shear_bound=4 if legacy_normalization else 30\n        result=least_squares(residual,theta,bounds=([-bound,-bound,-shape_bound,-shear_bound],[bound,bound,shape_bound,shear_bound]),\n                             loss=\'soft_l1\',f_scale=0.12,max_nfev=500)\n        fits.append((float(np.mean(residual(result.x)**2)),result))\n    _,result=min(fits,key=lambda item:item[0])\n    H=compose(result.x,normalizer)\n    world=project(quads,H)\n    _,_,lengths,_=measures(world)\n    # Give each track one vote; correlated video frames do not count as independent cars.\n    scale=length_m/np.median([np.median(lengths[g]) for g in groups])\n    H=np.diag([scale,scale,1])@H\n    world=project(quads,H)\n    directions=world[:,1]-world[:,0]\n    directions/=np.linalg.norm(directions,axis=1)[:,None]\n    directions[directions[:,1]<0]*=-1\n    direction=np.median(directions,axis=0)\n    angle=math.pi/2-math.atan2(direction[1],direction[0])\n    rotation=np.array([[math.cos(angle),-math.sin(angle),0],[math.sin(angle),math.cos(angle),0],[0,0,1]])\n    H=rotation@H\n    center=np.median(project(quads,H).mean(axis=1),axis=0)\n    H=np.array([[1,0,-center[0]],[0,1,-center[1]],[0,0,1]])@H\n    H/=np.linalg.norm(H[2,:2]) if np.linalg.norm(H[2,:2])>0 else H[2,2]\n    angle_error,_,lengths,widths=measures(project(quads,H))\n    singular=np.linalg.svd(result.jac,compute_uv=False)\n    diagnostics={\'fit_track_ids\':[int(i) for i in unique],\'fit_observations\':len(records),\n       \'median_angle_error_deg\':float(np.median(np.degrees(np.arcsin(np.clip(np.abs(angle_error),0,1))))),\n       \'median_length_m\':float(np.median(lengths)),\'median_width_m\':float(np.median(widths)),\n       \'parameter_condition_number\':float(singular[0]/max(singular[-1],1e-12)),\n       \'optimizer_success\':bool(result.success),\'theta\':result.x.tolist(),\n       \'dimension_prior_length_m\':length_m,\'dimension_prior_width_m\':width_m}\n    return H,diagnostics\n\n\ndef prepare(observations, indices, width, height, longitudinal_order=\'checkpoint\'):\n    by_id=defaultdict(list)\n    for r in observations:by_id[r[\'track_id\']].append(r)\n    records=[]\n    for tid,group in by_id.items():\n        group.sort(key=lambda r:r[\'time_s\'])\n        centers=np.array([[(r[\'baseline_box\'][0]+r[\'baseline_box\'][2])/2,\n                           (r[\'baseline_box\'][1]+r[\'baseline_box\'][3])/2] for r in group])\n        if len(group)<8 or np.linalg.norm(centers[-1]-centers[0])<80:continue\n        side=\'left\' if np.median(centers[:,0])<width/2 else \'right\'\n        for j,r in enumerate(group):\n            p=np.array(r[\'keypoints\'])[indices]\n            if p.shape[1]>2 and np.min(p[:,2])<0.30:continue\n            if r[\'geometry_confidence\']<0.25 or r[\'association_iou\']<0.40:continue\n            quad=p[:,:2]\n            if not np.isfinite(quad).all():continue\n            if longitudinal_order==\'motion\':\n                quad=quad[np.argsort(np.arctan2(*(quad-quad.mean(axis=0))[:,::-1].T))]\n            if not cv2.isContourConvex(quad.astype(np.float32)):continue\n            if abs(cv2.contourArea(quad.astype(np.float32)))<300:continue\n            if np.min(np.linalg.norm(np.roll(quad,-1,axis=0)-quad,axis=1))<8:continue\n            cx,cy=quad.mean(axis=0)\n            if not (0.20*height<cy<0.82*height and 0.02*width<cx<0.98*width):continue\n            if np.min(quad[:,0])<4 or np.max(quad[:,0])>width-4 or np.max(quad[:,1])>height-4:continue\n            if longitudinal_order==\'motion\':\n                lo=max(0,j-3);hi=min(len(group)-1,j+3)\n                motion=centers[hi]-centers[lo]\n                if np.linalg.norm(motion)<5:continue\n                edge=np.roll(quad,-1,axis=0)-quad\n                alignment=np.abs(edge@motion)/(np.linalg.norm(edge,axis=1)*np.linalg.norm(motion)+1e-9)\n                scores=[np.mean(alignment[[0,2]]),np.mean(alignment[[1,3]])]\n                if abs(scores[0]-scores[1])<0.08:continue\n                if scores[1]>scores[0]:quad=np.roll(quad,-1,axis=0)\n            elif longitudinal_order!=\'checkpoint\':\n                raise ValueError(\'Unknown longitudinal corner-order method.\')\n            records.append({\'track_id\':int(tid),\'time_s\':r[\'time_s\'],\'frame_index\':r[\'frame_index\'],\n                            \'side\':side,\'quad\':quad.tolist()})\n    return records\n\n\ndef estimate_track(records,H):\n    times=np.array([r[\'time_s\'] for r in records],dtype=float)\n    quads=np.array([r[\'quad\'] for r in records],dtype=float)\n    world=project(quads,H)\n    centers=world.mean(axis=1)\n    t=times-times.mean()\n    initial=np.r_[np.polyfit(t,centers[:,0],1),np.polyfit(t,centers[:,1],1)]\n    def residual(p):return np.r_[(p[0]*t+p[1]-centers[:,0]),(p[2]*t+p[3]-centers[:,1])]\n    fitted=least_squares(residual,initial,loss=\'soft_l1\',f_scale=0.4)\n    speed=float(np.hypot(fitted.x[0],fitted.x[2])*3.6)\n    rms=float(np.sqrt(np.mean(residual(fitted.x)**2)))\n    angles,_,lengths,widths=measures(world)\n    length_cv=float(1.4826*np.median(np.abs(lengths-np.median(lengths)))/np.median(lengths))\n    angle_error=float(np.median(np.degrees(np.arcsin(np.clip(np.abs(angles),0,1)))))\n    midpoint=(times[0]+times[-1])/2\n    half_speeds=[]\n    for mask in [times<=midpoint,times>midpoint]:\n        if mask.sum()>=4:\n            velocity=np.array([np.polyfit(times[mask]-times[mask].mean(),centers[mask,k],1)[0] for k in [0,1]])\n            half_speeds.append(float(np.linalg.norm(velocity)*3.6))\n    drift=abs(half_speeds[0]-half_speeds[1])/max(speed,1) if len(half_speeds)==2 else None\n    return {\'experimental_kmh\':speed,\'trajectory_fit_rms_m\':rms,\'length_relative_mad\':length_cv,\n            \'angle_error_deg\':angle_error,\'half_span_speed_relative_difference\':drift,\n            \'median_inferred_length_m\':float(np.median(lengths)),\'median_inferred_width_m\':float(np.median(widths))}\n\n\ndef sample_balanced(records, maximum=10):\n    groups=defaultdict(list)\n    for r in records:groups[r[\'track_id\']].append(r)\n    return [group[i] for group in groups.values() for i in np.unique(np.linspace(0,len(group)-1,min(len(group),maximum)).astype(int))]\n\n\ndef fit_road_mappings(records, report):\n    width,height=report[\'width\'],report[\'height\']\n    calibration={}\n    for side in [\'left\',\'right\']:\n        local=[r for r in records if r[\'side\']==side]\n        ids=sorted(set(r[\'track_id\'] for r in local))\n        if len(ids)<3:\n            calibration[side]={\'accepted\':False,\'reason\':\'Fewer than three qualifying car tracks\',\'track_ids\':ids}\n            continue\n        # Deterministic holdout by track identity; held-out predicted boxes are consistency evidence only.\n        held_ids=ids[::3]\n        fit_ids=[i for i in ids if i not in held_ids]\n        if len(fit_ids)<3:fit_ids=ids;held_ids=[]\n        fitting=sample_balanced([r for r in local if r[\'track_id\'] in fit_ids])\n        H,diagnostics=fit_mapping(fitting,width,height)\n        alternatives=[]\n        for length_m,width_m in DIMENSION_SCENARIOS[1:]:\n            alternative,_=fit_mapping(fitting,width,height,length_m,width_m,fast=True)\n            alternatives.append(alternative)\n        rng=np.random.default_rng(42)\n        for _ in range(10):\n            selected=set(rng.choice(fit_ids,size=max(3,round(len(fit_ids)*0.75)),replace=False).tolist())\n            subset=[r for r in fitting if r[\'track_id\'] in selected]\n            alternative,_=fit_mapping(subset,width,height,fast=True)\n            alternatives.append(alternative)\n        held=[r for r in local if r[\'track_id\'] in held_ids]\n        held_angle=None\n        if held:\n            angles,_,_,_=measures(project(np.array([r[\'quad\'] for r in held]),H))\n            held_angle=float(np.median(np.degrees(np.arcsin(np.clip(np.abs(angles),0,1)))))\n        accepted=diagnostics[\'median_angle_error_deg\']<20 and (held_angle is None or held_angle<25)\n        accepted=accepted and report[\'ground_order_vote_agreement\']>=0.8 and diagnostics[\'parameter_condition_number\']<1e5\n        diagnostics.update({\'accepted\':bool(accepted),\'held_out_track_ids\':held_ids,\n          \'held_out_median_angle_error_deg\':held_angle,\'H_image_to_experimental_meters\':H.tolist(),\n          \'alternative_mappings\':[a.tolist() for a in alternatives],\n          \'interpretation\':\'Approximate local plane and dimension-prior mapping; no measured accuracy.\'})\n        calibration[side]=diagnostics\n        diagnostics[\'support_hull_image\'] = cv2.convexHull(\n            np.array([r[\'quad\'] for r in local], dtype=np.float32).reshape(-1, 2)).reshape(-1, 2).tolist()\n    return calibration\n\n\ndef calculate_speed_rows(records, calibration):\n    speed_rows=[]\n    for side in [\'left\',\'right\']:\n        diagnostics=calibration.get(side,{})\n        if not diagnostics.get(\'accepted\'):\n            continue\n        H=np.array(diagnostics[\'H_image_to_experimental_meters\'])\n        alternatives=[np.array(a) for a in diagnostics[\'alternative_mappings\']]\n        hull=np.array(diagnostics[\'support_hull_image\'], dtype=np.float32)\n        # Restrict reuse to the road region supported by the calibration footprints.\n        local=[r for r in records if r[\'side\']==side and all(\n            cv2.pointPolygonTest(hull, (float(p[0]),float(p[1])), True)>=-0.01 for p in r[\'quad\'])]\n        ids=sorted(set(r[\'track_id\'] for r in local))\n        for tid in ids:\n            group=sorted([r for r in local if r[\'track_id\']==tid],key=lambda r:r[\'time_s\'])\n            if len(group)<8 or group[-1][\'time_s\']-group[0][\'time_s\']<1.2:continue\n            nominal=estimate_track(group,H)\n            values=[nominal[\'experimental_kmh\']]+[estimate_track(group,a)[\'experimental_kmh\'] for a in alternatives]\n            low,high=float(min(values)),float(max(values))\n            sensitivity=(high-low)/max(nominal[\'experimental_kmh\'],1)\n            reasons=[]\n            if nominal[\'length_relative_mad\']>0.25:reasons.append(\'predicted_size_inconsistent\')\n            if nominal[\'trajectory_fit_rms_m\']>0.8:reasons.append(\'trajectory_not_consistent_with_local_plane\')\n            if nominal[\'angle_error_deg\']>25:reasons.append(\'footprint_not_rectangular_after_mapping\')\n            if nominal[\'half_span_speed_relative_difference\'] is not None and nominal[\'half_span_speed_relative_difference\']>0.35:reasons.append(\'speed_changes_or_geometry_drift_across_span\')\n            if sensitivity>0.8:reasons.append(\'high_calibration_sensitivity\')\n            speed_rows.append({\'track_id\':tid,\'side\':side,\'time_start_s\':group[0][\'time_s\'],\n              \'time_end_s\':group[-1][\'time_s\'],\'geometry_observations\':len(group),\n              \'quality_passed\':not bool(reasons),\'quality_reasons\':\';\'.join(reasons),\n              \'dimension_and_fit_sensitivity_low_kmh\':low,\'dimension_and_fit_sensitivity_high_kmh\':high,\n              **nominal})\n    return speed_rows\n\n\ndef run(calibration=None, geometry_dir=None, output_dir=None, calibration_reused=None):\n    geometry_dir=Path(geometry_dir) if geometry_dir is not None else GEOMETRY\n    output_dir=Path(output_dir) if output_dir is not None else OUT\n    output_dir.mkdir(parents=True,exist_ok=True)\n    report=json.loads((geometry_dir/\'geometry_run_report.json\').read_text())\n    observations=json.loads((geometry_dir/\'observations.json\').read_text())\n    records=prepare(observations,report[\'ground_indices\'],report[\'width\'],report[\'height\'])\n    reused=(calibration is not None) if calibration_reused is None else calibration_reused\n    if calibration is None:\n        calibration=fit_road_mappings(records, report)\n    before_support_filter=len(records)\n    records=[r for r in records if calibration.get(r[\'side\'],{}).get(\'accepted\') and all(\n        cv2.pointPolygonTest(np.array(calibration[r[\'side\']][\'support_hull_image\'],dtype=np.float32),\n                             (float(p[0]),float(p[1])),True)>=-0.01 for p in r[\'quad\'])]\n    speed_rows=calculate_speed_rows(records, calibration)\n    (output_dir/\'filtered_ground_footprints.json\').write_text(json.dumps(records),encoding=\'utf-8\')\n    (output_dir/\'calibration.json\').write_text(json.dumps(calibration,indent=2),encoding=\'utf-8\')\n    fields=[\'track_id\',\'side\',\'time_start_s\',\'time_end_s\',\'geometry_observations\',\n            \'quality_passed\',\'quality_reasons\',\'dimension_and_fit_sensitivity_low_kmh\',\n            \'dimension_and_fit_sensitivity_high_kmh\',\'experimental_kmh\',\'trajectory_fit_rms_m\',\n            \'length_relative_mad\',\'angle_error_deg\',\'half_span_speed_relative_difference\',\n            \'median_inferred_length_m\',\'median_inferred_width_m\']\n    for name, rows in [(\'experimental_speeds.csv\',speed_rows),\n                       (\'qualified_experimental_speeds.csv\',[r for r in speed_rows if r[\'quality_passed\']])]:\n        with (output_dir/name).open(\'w\',newline=\'\',encoding=\'utf-8\') as f:\n            writer=csv.DictWriter(f,fieldnames=fields)\n            writer.writeheader()\n            writer.writerows(rows)\n    api_results=[{\'track_id\':r[\'track_id\'],\'carriageway\':r[\'side\'],\n      \'time_interval_s\':[r[\'time_start_s\'],r[\'time_end_s\']],\n      \'experimental_kmh\':r[\'experimental_kmh\'],\n      \'estimated_kmh\':r[\'experimental_kmh\'],\n      \'sensitivity_range_kmh\':[r[\'dimension_and_fit_sensitivity_low_kmh\'],r[\'dimension_and_fit_sensitivity_high_kmh\']],\n      \'range_type\':\'dimension_and_calibration_sensitivity\',\'accuracy_validated\':False,\n      \'method\':\'learned_3d_ground_footprints_with_assumed_vehicle_dimensions\'}\n      for r in speed_rows if r[\'quality_passed\']]\n    (output_dir/\'vehicle_speed_estimates.json\').write_text(json.dumps(api_results,indent=2),encoding=\'utf-8\')\n    result={\'calibration_reused\':reused,\'source_geometry_observations\':len(observations),\'filtered_ground_footprints\':len(records),\n      \'footprints_outside_usable_calibration\':before_support_filter-len(records),\n      \'geometry_model_filename\':report[\'model_filename\'],\'geometry_model_sha256\':report[\'model_sha256\'],\n      \'ground_corner_indices\':report[\'ground_indices\'],\n      \'experimental_track_estimates\':len(speed_rows),\'quality_passed_track_estimates\':sum(r[\'quality_passed\'] for r in speed_rows),\n      \'nominal_car_length_m\':LENGTH_M,\'nominal_car_width_m\':WIDTH_M,\'dimension_scenarios_m\':DIMENSION_SCENARIOS,\n      \'range_type\':\'Dimension-prior and track-subset sensitivity envelope. Not a statistical confidence interval.\',\n      \'scope\':\'Local visible road regions on each carriageway; passenger-car hypotheses only.\',\n      \'accuracy_validated\':False,\'speed_units\':\'km/h under assumed vehicle dimensions\',\n      \'limitations\':[\'No independent speed or road-distance ground truth.\',\n        \'Ground corners are model predictions; hidden corners can be wrong.\',\n        \'A local planar road and approximately constant speed over each fitted span are assumed.\',\n        \'Dimension priors are explicit analyst assumptions, not recognized make/model dimensions.\',\n        \'Sensitivity range does not bound unmodelled errors such as lens distortion or road grade.\']}\n    (output_dir/\'speed_run_report.json\').write_text(json.dumps(result,indent=2),encoding=\'utf-8\')\n    print(json.dumps(result,indent=2))\n    for r in speed_rows:print(r[\'track_id\'],r[\'side\'],round(r[\'experimental_kmh\'],1),r[\'quality_passed\'],r[\'quality_reasons\'])\n    return calibration,speed_rows,records\n\n\nif __name__==\'__main__\':run()\n', 'camera_cache.py': '"""Conservative camera-view checks and a versioned, integrity-checked calibration cache."""\nimport hashlib\nimport json\nimport re\nfrom pathlib import Path\n\nimport cv2\nimport numpy as np\n\nSCHEMA_VERSION = 1\n\n\ndef file_sha256(path):\n    digest = hashlib.sha256()\n    with Path(path).open(\'rb\') as stream:\n        for chunk in iter(lambda: stream.read(1024 * 1024), b\'\'):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef canonical_hash(value):\n    return hashlib.sha256(json.dumps(value, sort_keys=True, allow_nan=False).encode()).hexdigest()\n\n\ndef read_views(video, start_s, duration_s):\n    capture = cv2.VideoCapture(str(video))\n    try:\n        if not capture.isOpened():\n            raise ValueError(\'Cannot decode source video.\')\n        width = int(capture.get(cv2.CAP_PROP_FRAME_WIDTH))\n        height = int(capture.get(cv2.CAP_PROP_FRAME_HEIGHT))\n        fps = float(capture.get(cv2.CAP_PROP_FPS))\n        frames = int(capture.get(cv2.CAP_PROP_FRAME_COUNT))\n        if fps <= 0 or min(width, height, frames) <= 0:\n            raise ValueError(\'Invalid video metadata.\')\n        start, end = round(start_s * fps), round((start_s + duration_s) * fps)\n        if start < 0 or end <= start or end > frames:\n            raise ValueError(\'Requested interval is outside the source video.\')\n        views = []\n        indices = np.unique(np.linspace(start, end - 1, 5).astype(int))\n        for index in indices:\n            capture.set(cv2.CAP_PROP_POS_FRAMES, int(index))\n            ok, frame = capture.read()\n            if not ok:\n                raise ValueError(f\'Cannot decode view-check frame {index}.\')\n            gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)\n            views.append(cv2.resize(gray, (960, round(height * 960 / width))))\n        return {\'width\': width, \'height\': height, \'fps\': fps, \'frames\': frames,\n                \'start_frame\': start, \'end_frame\': end,\n                \'view_check_frames\': indices.tolist()}, views\n    finally:\n        capture.release()\n\n\ndef compare_views(reference, current):\n    """Reject uncertain matches as well as sufficiently large shifts, zooms or rotations.\n\n    This is a sampled visual consistency check, not proof that the camera never moved.\n    Exclude timestamps, borders and foreground overlays; RANSAC discounts moving vehicles.\n    """\n    if reference is None or current is None or reference.shape != current.shape:\n        return {\'valid\': False, \'reason\': \'view_image_shape_changed\'}\n    height, width = reference.shape\n    mask = np.zeros_like(reference, dtype=np.uint8)\n    mask[round(.15 * height):round(.88 * height), round(.03 * width):round(.97 * width)] = 255\n    detector = cv2.ORB_create(nfeatures=5000, fastThreshold=12)\n    kp1, desc1 = detector.detectAndCompute(reference, mask)\n    kp2, desc2 = detector.detectAndCompute(current, mask)\n    if desc1 is None or desc2 is None:\n        return {\'valid\': False, \'reason\': \'insufficient_static_features\'}\n    pairs = cv2.BFMatcher(cv2.NORM_HAMMING).knnMatch(desc1, desc2, k=2)\n    matches = [a for pair in pairs if len(pair) == 2 for a, b in [pair] if a.distance < .7 * b.distance]\n    if len(matches) < 25:\n        return {\'valid\': False, \'reason\': \'insufficient_view_matches\', \'matches\': len(matches)}\n    a = np.float32([kp1[m.queryIdx].pt for m in matches])\n    b = np.float32([kp2[m.trainIdx].pt for m in matches])\n    matrix, inliers = cv2.findHomography(a, b, cv2.RANSAC, 3.0)\n    if matrix is None or inliers is None or not np.isfinite(matrix).all():\n        return {\'valid\': False, \'reason\': \'view_alignment_failed\'}\n    selected = inliers.ravel().astype(bool)\n    count = int(selected.sum())\n    coverage = min(abs(cv2.contourArea(cv2.convexHull(p[selected]))) / (width * height)\n                   for p in [a, b]) if count >= 3 else 0\n    corners = np.float32([[0, 0], [width, 0], [width, height], [0, height]])\n    projected = cv2.perspectiveTransform(corners[None], matrix)[0]\n    displacement = float(np.max(np.linalg.norm((projected - corners) / [width, height], axis=1)))\n    valid = count >= 25 and count / len(matches) >= .5 and coverage >= .10 and displacement <= .012\n    return {\'valid\': bool(valid), \'reason\': \'consistent_view\' if valid else \'view_changed_or_uncertain\',\n            \'matches\': len(matches), \'inliers\': count, \'feature_coverage_fraction\': coverage,\n            \'max_normalized_corner_displacement\': displacement}\n\n\ndef stable_view(views):\n    comparisons = [compare_views(views[0], view) for view in views[1:]]\n    return {\'valid\': bool(comparisons) and all(item[\'valid\'] for item in comparisons),\n            \'comparisons\': comparisons,\n            \'limitation\': \'Five sampled frames cannot exclude brief camera movement between samples.\'}\n\n\ndef valid_calibration(calibration):\n    if not isinstance(calibration, dict):\n        return False\n    accepted = 0\n    try:\n        for side in [\'left\', \'right\']:\n            mapping = calibration.get(side, {})\n            if not mapping.get(\'accepted\'):\n                continue\n            accepted += 1\n            matrices = [mapping[\'H_image_to_experimental_meters\'], *mapping[\'alternative_mappings\']]\n            if len(matrices) < 2:\n                return False\n            for value in matrices:\n                matrix = np.array(value, dtype=float)\n                if matrix.shape != (3, 3) or not np.isfinite(matrix).all() or abs(np.linalg.det(matrix)) < 1e-12:\n                    return False\n            hull = np.array(mapping[\'support_hull_image\'], dtype=np.float32)\n            if hull.ndim != 2 or hull.shape[1] != 2 or len(hull) < 3 or not np.isfinite(hull).all():\n                return False\n            if not cv2.isContourConvex(hull) or abs(cv2.contourArea(hull)) < 300:\n                return False\n    except (KeyError, TypeError, ValueError, cv2.error):\n        return False\n    return accepted > 0\n\n\nclass CalibrationCache:\n    def __init__(self, directory, camera_id):\n        if not camera_id.strip():\n            raise ValueError(\'A nonempty camera ID is required.\')\n        slug = re.sub(r\'[^a-zA-Z0-9_-]+\', \'-\', camera_id).strip(\'-\')[:48] or \'camera\'\n        self.camera_id = camera_id\n        self.directory = Path(directory) / (slug + \'-\' + canonical_hash(camera_id)[:12])\n        self.path = self.directory / \'calibration_cache.json\'\n\n    def load(self, signature, views):\n        if not self.path.exists():\n            return None, {\'status\': \'miss\', \'reason\': \'no_saved_calibration\'}\n        try:\n            entry = json.loads(self.path.read_text(encoding=\'utf-8\'))\n            if entry[\'schema_version\'] != SCHEMA_VERSION or entry[\'camera_id\'] != self.camera_id:\n                return None, {\'status\': \'miss\', \'reason\': \'cache_schema_or_camera_changed\'}\n            if entry[\'signature\'] != signature:\n                return None, {\'status\': \'miss\', \'reason\': \'resolution_model_or_calibration_settings_changed\'}\n            if canonical_hash(entry[\'calibration\']) != entry[\'calibration_sha256\'] or not valid_calibration(entry[\'calibration\']):\n                return None, {\'status\': \'miss\', \'reason\': \'invalid_calibration_data\'}\n            view_path = self.directory / \'reference_view.png\'\n            if file_sha256(view_path) != entry[\'reference_sha256\']:\n                return None, {\'status\': \'miss\', \'reason\': \'invalid_reference_image\'}\n            reference = cv2.imread(str(view_path), cv2.IMREAD_GRAYSCALE)\n            checks = [compare_views(reference, current) for current in views]\n            if not all(check[\'valid\'] for check in checks):\n                return None, {\'status\': \'miss\', \'reason\': \'camera_view_changed_or_uncertain\', \'view_checks\': checks}\n            return entry[\'calibration\'], {\'status\': \'hit\', \'reason\': \'saved_calibration_matches_camera_view\',\n                                          \'view_checks\': checks}\n        except (OSError, ValueError, KeyError, TypeError, cv2.error):\n            return None, {\'status\': \'miss\', \'reason\': \'cache_unreadable_or_corrupt\'}\n\n    def save(self, signature, reference, calibration, provenance):\n        if not valid_calibration(calibration):\n            raise ValueError(\'No usable road mapping; calibration will not be cached.\')\n        self.directory.mkdir(parents=True, exist_ok=True)\n        view_path = self.directory / \'reference_view.png\'\n        if not cv2.imwrite(str(view_path), reference):\n            raise OSError(\'Could not save camera reference.\')\n        entry = {\'schema_version\': SCHEMA_VERSION, \'camera_id\': self.camera_id,\n                 \'signature\': signature, \'calibration\': calibration,\n                 \'calibration_sha256\': canonical_hash(calibration),\n                 \'reference_sha256\': file_sha256(view_path), \'provenance\': provenance,\n                 \'accuracy_validated\': False}\n        temporary = self.path.with_suffix(\'.tmp\')\n        temporary.write_text(json.dumps(entry, indent=2, allow_nan=False), encoding=\'utf-8\')\n        temporary.replace(self.path)\n', 'benchmark_speed.py': '"""Label-independent single-pass estimators; reference speeds are used only by evaluation/tuning."""\nfrom collections import defaultdict\nfrom pathlib import Path\nimport json\nimport math\n\nimport cv2\nimport numpy as np\nfrom scipy.optimize import least_squares\nimport calibrate_speed as geometry\n\n\ndef target_records(observations, report, min_height=0, max_height=10000, corner_order=\'checkpoint\'):\n    prepared=geometry.prepare(observations,report[\'ground_indices\'],report[\'width\'],report[\'height\'],corner_order)\n    boxes={(r[\'track_id\'],r[\'frame_index\']):r[\'baseline_box\'] for r in observations}\n    groups=defaultdict(list)\n    for row in prepared:\n        box=boxes[row[\'track_id\'],row[\'frame_index\']]\n        if min_height<=box[3]-box[1]<=max_height:groups[row[\'track_id\']].append(row)\n    if not groups:return []\n    # The moving pass with the largest observed footprint is the target, independent of speed labels.\n    target=max(groups.values(),key=lambda rows:max(abs(cv2.contourArea(np.array(r[\'quad\'],np.float32))) for r in rows))\n    return sorted(target,key=lambda r:r[\'time_s\'])\n\n\ndef estimate_single_pass(observations, report, config):\n    rows=target_records(observations,report,config.get(\'min_height\',0),config.get(\'max_height\',10000),config.get(\'corner_order\',\'checkpoint\'))\n    if len(rows)<8 or rows[-1][\'time_s\']-rows[0][\'time_s\']<.25:\n        return {\'available\':False,\'reason\':\'insufficient_unclipped_ground_geometry\',\'observations\':len(rows)}\n    length=config.get(\'length_m\',4.5);width=config.get(\'width_m\',1.8)\n    H,diagnostics=geometry.fit_mapping(rows,report[\'width\'],report[\'height\'],length,width,\n                                      legacy_normalization=config.get(\'legacy_normalization\',False))\n    calculated=geometry.estimate_track(rows,H)\n    result={\'available\':True,\'raw_kmh\':calculated[\'experimental_kmh\'],\n            \'estimated_kmh\':calculated[\'experimental_kmh\']*config.get(\'scale\',1),\n            \'time_interval_s\':[rows[0][\'time_s\'],rows[-1][\'time_s\']],\n            \'observations\':len(rows),\'track_id\':rows[0][\'track_id\'],\n            \'mapping\':H.tolist(),\'diagnostics\':diagnostics,\'motion_diagnostics\':calculated,\n            \'support_hull_image\':cv2.convexHull(np.array([r[\'quad\'] for r in rows],np.float32).reshape(-1,2)).reshape(-1,2).tolist(),\n            \'method\':\'single_pass_rectangle_calibration\',\'reference_speed_used\':False}\n    if config.get(\'method\')==\'projective_delay\':\n        delay=projective_delay(rows,length)\n        if not delay[\'available\']:return delay\n        result.update(delay)\n        result[\'estimated_kmh\']=result[\'raw_kmh\']*config.get(\'scale\',1)\n    result[\'calibration_length_m\']=length\n    result[\'scale_correction\']=config.get(\'scale\',1)\n    return result\n\n\ndef projective_delay(rows,length_m):\n    """Match front/rear ground positions through a shared 1-D projective motion curve.\n\n    For constant speed, the rear follows the front\'s path after a delay L/v.\n    Fitting this delay estimates speed without reading a reference speed or clip filename.\n    """\n    quads=np.array([r[\'quad\'] for r in rows]);centers=quads.mean(axis=1)\n    direction=centers[-1]-centers[0];direction/=np.linalg.norm(direction)\n    front=(quads[:,1]+quads[:,2])/2;rear=(quads[:,0]+quads[:,3])/2\n    if np.median((front-rear)@direction)<0:front,rear=rear,front\n    front=front@direction;rear=rear@direction\n    times=np.array([r[\'time_s\'] for r in rows]);span=times[-1]-times[0]\n    t=(times-times[0])/span\n    origin=np.mean(np.r_[front,rear]);normalizer=max(np.ptp(np.r_[front,rear]),1)\n    front=(front-origin)/normalizer;rear=(rear-origin)/normalizer\n    linear=np.polyfit(t,front,1)\n\n    def residual(parameters):\n        a,b,c,log_delay=parameters;delay=np.exp(log_delay)\n        def prediction(s):\n            den=1+c*s\n            return (a+b*s)/np.maximum(den,.02),np.maximum(.03-den,0)*10\n        f,fp=prediction(t);r,rp=prediction(t-delay)\n        return np.r_[f-front,r-rear,fp,rp]\n    fits=[]\n    for c in [-.8,-.4,0,.5,1.5]:\n        initial=[linear[1],linear[0],c,math.log(.25/span)]\n        fit=least_squares(residual,initial,bounds=([-5,-20,-.95,math.log(.03/span)],\n                                                 [5,20,4,math.log(4/span)]),loss=\'soft_l1\',f_scale=.02,max_nfev=500)\n        fits.append((np.mean(residual(fit.x)**2),fit))\n    error,fit=min(fits,key=lambda pair:pair[0]);delay_s=float(np.exp(fit.x[3])*span)\n    return {\'available\':True,\'raw_kmh\':length_m/delay_s*3.6,\'method\':\'projective_delay\',\n            \'projective_delay_s\':delay_s,\'projective_fit_rms_pixels\':float(np.sqrt(error)*normalizer),\n            \'projective_parameters\':fit.x.tolist()}\n\n\ndef load_clip(directory):\n    directory=Path(directory)\n    observations=json.loads((directory/\'observations.json\').read_text())\n    report=json.loads((directory/\'geometry_run_report.json\').read_text())\n    # Recompute corner-order voting from observations, including early pilot outputs.\n    points=np.array([r[\'keypoints\'] for r in observations])\n    if len(points):\n        vote=points[:,:4,1].mean(axis=1)>points[:,4:,1].mean(axis=1)\n        report[\'ground_indices\']=list(range(4)) if vote.mean()>=.5 else list(range(4,8))\n        report[\'ground_order_vote_agreement\']=float(max(vote.mean(),1-vote.mean()))\n    return observations,report\n', 'reference_calibration.py': '"""Reference-assisted road calibration and verified transfer between overlapping road views."""\nfrom pathlib import Path\nfrom types import SimpleNamespace\nimport cv2\nimport numpy as np\nfrom scipy.optimize import least_squares,linprog\n\nimport calibrate_speed as geo\nfrom benchmark_speed import target_records\n\n\ndef fit_motion(world,times):\n    t=times-times.mean()\n    initial=np.r_[np.polyfit(t,world[:,0],1),np.polyfit(t,world[:,1],1)]\n    def residual(p):return np.r_[p[0]*t+p[1]-world[:,0],p[2]*t+p[3]-world[:,1]]\n    # Use the same constant-velocity least-squares estimator for fitting and\n    # evaluation. Switching estimators after calibration changes its scale.\n    return SimpleNamespace(x=initial),residual(initial)\n\n\ndef refine_metric_scale(estimates,references):\n    """Choose one physical distance-scale multiplier that minimizes worst pilot error."""\n    estimates=np.asarray(estimates,dtype=float);references=np.asarray(references,dtype=float)\n    A=np.vstack([np.c_[estimates,-np.ones(len(estimates))],np.c_[-estimates,-np.ones(len(estimates))]])\n    b=np.r_[references,-references]\n    result=linprog([0,1],A_ub=A,b_ub=b,bounds=[(.1,10),(0,None)],method=\'highs\')\n    if not result.success:raise ValueError(\'Pilot metric-scale refinement failed.\')\n    return float(result.x[0]),float(result.x[1])\n\n\ndef register_road(reference,current):\n    """Find the ground-plane transform using foreground road texture, not speed labels."""\n    reference=cv2.resize(reference,(960,540));current=cv2.resize(current,(960,540))\n    if reference.ndim==3:reference=cv2.cvtColor(reference,cv2.COLOR_BGR2GRAY)\n    if current.ndim==3:current=cv2.cvtColor(current,cv2.COLOR_BGR2GRAY)\n    mask=np.zeros((540,960),np.uint8);mask[324:,240:941]=255\n    detector=cv2.SIFT_create(nfeatures=8000,contrastThreshold=.01)\n    kr,dr=detector.detectAndCompute(reference,mask);kc,dc=detector.detectAndCompute(current,mask)\n    if dr is None or dc is None:return None,{\'accepted\':False,\'reason\':\'no_road_features\'}\n    pairs=cv2.BFMatcher().knnMatch(dc,dr,k=2)\n    matches=[a for pair in pairs if len(pair)==2 for a,b in [pair] if a.distance<.7*b.distance]\n    if len(matches)<12:return None,{\'accepted\':False,\'reason\':\'insufficient_road_matches\',\'matches\':len(matches)}\n    source=np.float32([kc[m.queryIdx].pt for m in matches]);dest=np.float32([kr[m.trainIdx].pt for m in matches])\n    H,inliers=cv2.findHomography(source,dest,cv2.RANSAC,2)\n    if H is None or inliers is None:return None,{\'accepted\':False,\'reason\':\'road_alignment_failed\'}\n    selected=inliers.ravel().astype(bool);count=int(selected.sum())\n    coverage=min(abs(cv2.contourArea(cv2.convexHull(p[selected])))/(960*540) for p in [source,dest]) if count>=3 else 0\n    predicted=geo.project(source[selected],H)\n    rms=float(np.sqrt(np.mean(np.sum((predicted-dest[selected])**2,axis=1))))\n    accepted=count>=12 and count/len(matches)>=.5 and coverage>=.015 and rms<=2\n    evidence={\'accepted\':bool(accepted),\'matches\':len(matches),\'inliers\':count,\'road_coverage\':coverage,\n              \'alignment_rms_pixels_at_960\':rms,\'reason\':\'verified_road_overlap\' if accepted else \'uncertain_road_alignment\'}\n    scale=np.diag([.5,.5,1])\n    return (np.linalg.inv(scale)@H@scale if accepted else None),evidence\n\n\ndef trajectory_points(observations,report):\n    rows=target_records(observations,report,80,500)\n    if len(rows)<8:return None,None,rows\n    quads=np.array([r[\'quad\'] for r in rows])\n    # Nearest face\'s ground midpoint avoids averaging unreliable hidden rear corners.\n    points=(quads[:,0]+quads[:,3])/2\n    times=np.array([r[\'time_s\'] for r in rows])\n    return points,times,rows\n\n\ndef fit_reference_mapping(clips):\n    """Fit one local projective metric mapping using designated pilot speeds only.\n\n    Each item has aligned points, times, quads and an explicitly supplied pilot reference.\n    Results on these same pilots are fitting diagnostics, not independent accuracy evidence.\n    """\n    records=[row for clip in clips for row in clip[\'rows\']]\n    quads=np.array([r[\'quad\'] for r in records]);center=np.median(quads.reshape(-1,2),axis=0)\n    N=np.array([[2/1920,0,-2*center[0]/1920],[0,2/1920,-2*center[1]/1920],[0,0,1]])\n    _,initial=geo.fit_mapping(records,1920,1080,4.9,1.87)\n\n    def mapping(parameters):\n        scale=np.exp(parameters[4])\n        return np.diag([scale,scale,1])@geo.compose(parameters[:4],N)\n\n    def residual(parameters):\n        H=mapping(parameters);out=[]\n        for clip in clips:\n            world=geo.project(clip[\'points\'],H);t=clip[\'times\']-clip[\'times\'].mean()\n            motion,_=fit_motion(world,clip[\'times\'])\n            velocity=motion.x[[0,2]]\n            prediction=world.mean(axis=0)+t[:,None]*velocity\n            target=clip[\'reference_kmh\']/3.6\n            out.extend(((world-prediction)/max(target,1)*3).ravel())\n            out.append((np.linalg.norm(velocity)-target)*50)\n        angles,parallel,lengths,widths=geo.measures(geo.project(quads,H))\n        # Predicted cuboids are approximate. Keep their shape as a weak regularizer.\n        out.extend((angles*.1).ravel())\n        out.extend((parallel*.1).ravel())\n        den=(np.c_[quads.reshape(-1,2),np.ones(len(quads)*4)]@H.T)[:,2]\n        out.extend(np.maximum(.03-den,0)*10)\n        return np.array(out)\n\n    fits=[]\n    for scale in [1,3,10,30]:\n        parameters=initial[\'theta\']+[np.log(scale)]\n        fit=least_squares(residual,parameters,bounds=([-100,-100,-7,-30,-5],[100,100,7,30,10]),\n                          loss=\'soft_l1\',f_scale=.1,max_nfev=1000)\n        fits.append((float(np.mean(residual(fit.x)**2)),fit))\n    error,fit=min(fits,key=lambda item:item[0]);H=mapping(fit.x)\n    raw=[]\n    for clip in clips:\n        motion,_=fit_motion(geo.project(clip[\'points\'],H),clip[\'times\'])\n        raw.append(float(np.hypot(motion.x[0],motion.x[2])*3.6))\n    scale,max_error=refine_metric_scale(raw,[c[\'reference_kmh\'] for c in clips])\n    H=np.diag([scale,scale,1])@H\n    support=cv2.convexHull(np.concatenate([c[\'points\'] for c in clips]).astype(np.float32)).reshape(-1,2)\n    return {\'H_image_to_meters\':H.tolist(),\'support_hull_image\':support.tolist(),\n            \'fit_rms\':float(np.sqrt(error)),\'parameters\':fit.x.tolist(),\n            \'reference_videos\':[c[\'video\'] for c in clips],\n            \'reference_speeds_kmh\':[c[\'reference_kmh\'] for c in clips],\n            \'metric_scale_refinement\':{\'multiplier\':scale,\'objective\':\'minimum worst absolute pilot speed error\',\n                                       \'maximum_pilot_error_kmh\':max_error,\'unrefined_pilot_estimates_kmh\':raw},\n            \'method\':\'reference_assisted_local_projective_calibration\',\n            \'physical_accuracy_verified\':False,\'scope\':\'Registered road region; this vehicle and recording setup.\'}\n\n\ndef estimate_reference_speed(points,times,mapping):\n    world=geo.project(points,np.array(mapping[\'H_image_to_meters\']))\n    fit,residual=fit_motion(world,times)\n    return {\'estimated_kmh\':float(np.hypot(fit.x[0],fit.x[2])*3.6),\n            \'trajectory_rms_m\':float(np.sqrt(np.mean(residual**2))),\n            \'time_interval_s\':[float(times[0]),float(times[-1])],\n            \'estimator_reads_reference_label\':False}\n', 'validation_experiment.py': '"""Evaluate a frozen pilot calibration on separate constant-speed clips and render paired videos."""\nimport argparse\nimport csv\nimport hashlib\nimport json\nfrom pathlib import Path\nimport shutil\nimport subprocess\n\nimport cv2\nimport numpy as np\nimport matplotlib\nmatplotlib.use(\'Agg\')\nimport matplotlib.pyplot as plt\n\nfrom benchmark_speed import load_clip,estimate_single_pass,target_records\nfrom camera_cache import file_sha256\nfrom calibrate_speed import project\nfrom reference_calibration import register_road,trajectory_points,estimate_reference_speed\n\nPILOT_NAMES={f\'MercedesAMG550_{value}.MP4\' for value in [30,60,100]}\nROOT=Path(__file__).resolve().parent\nAUTOMATIC_CONFIG={\'length_m\':4.9,\'width_m\':1.87,\'min_height\':100,\'max_height\':500,\n                  \'corner_order\':\'checkpoint\',\'legacy_normalization\':False}\nLEGACY_CONFIG={\'length_m\':4.5,\'width_m\':1.8,\'corner_order\':\'motion\',\'legacy_normalization\':True}\n\n\ndef predict(observations,report,image,reference,calibration):\n    """No filename or reference speed is available to this function."""\n    points,times,rows=trajectory_points(observations,report)\n    transform,evidence=register_road(reference,image)\n    if transform is not None and points is not None:\n        aligned=project(points,transform)\n        hull=np.array(calibration[\'support_hull_image\'],np.float32)\n        keep=np.array([cv2.pointPolygonTest(hull,tuple(map(float,p)),True)>=-10 for p in aligned])\n        if keep.sum()>=8 and times[keep][-1]-times[keep][0]>=.25:\n            result=estimate_reference_speed(aligned[keep],times[keep],calibration)\n            result.update(available=True,method=\'reference_assisted_registered_road\',\n                          track_id=rows[0][\'track_id\'],mapping=(np.array(calibration[\'H_image_to_meters\'])@transform).tolist(),\n                          support_hull_image=project(calibration[\'support_hull_image\'],np.linalg.inv(transform)).tolist(),\n                          road_registration=evidence,observations=int(keep.sum()))\n            result.update(calibration_reused=True,calibration_lookup_reason=\'registered_road_and_supported_vehicle_path\')\n            return result\n    result=estimate_single_pass(observations,report,AUTOMATIC_CONFIG)\n    result[\'road_registration\']=evidence\n    result[\'method\']=\'automatic_single_pass_geometry\'\n    result.update(calibration_reused=False,calibration_lookup_reason=\n                  \'vehicle_outside_saved_support\' if evidence.get(\'accepted\') else \'no_verified_saved_road_match\')\n    if result.get(\'available\'):\n        result[\'trajectory_rms_m\']=result[\'motion_diagnostics\'][\'trajectory_fit_rms_m\']\n    return result\n\n\ndef statistics(rows,column):\n    values=[r for r in rows if r.get(column) is not None and np.isfinite(r[column])]\n    if not values:return {\'clips\':len(rows),\'estimated\':0,\'coverage\':0}\n    errors=np.array([r[column]-r[\'reference_kmh\'] for r in values]);absolute=np.abs(errors)\n    return {\'clips\':len(rows),\'estimated\':len(values),\'coverage\':len(values)/max(len(rows),1),\n            \'mae_kmh\':float(absolute.mean()),\'rmse_kmh\':float(np.sqrt(np.mean(errors**2))),\n            \'bias_kmh\':float(errors.mean()),\'median_absolute_error_kmh\':float(np.median(absolute)),\n            \'maximum_absolute_error_kmh\':float(absolute.max()),\n            \'mape_percent\':float(np.mean([abs(r[column]-r[\'reference_kmh\'])/r[\'reference_kmh\']*100 for r in values])),\n            \'within_1_kmh\':int((absolute<=1).sum()),\'within_2_kmh\':int((absolute<=2).sum()),\n            \'errors_over_10_kmh\':int((absolute>10).sum())}\n\n\ndef grid(frame,H,hull):\n    hull=np.array(hull,np.float32)\n    if len(hull)<3:return frame\n    world=project(hull,H)\n    if not np.isfinite(world).all() or np.ptp(world,axis=0).max()>500:return frame\n    low,high=np.percentile(world,[5,95],axis=0)\n    mask=np.zeros(frame.shape[:2],np.uint8);cv2.fillConvexPoly(mask,hull.astype(int),255)\n    overlay=np.zeros_like(frame);inverse=np.linalg.inv(H)\n    for axis in [0,1]:\n        for value in np.arange(np.floor(low[axis]/5)*5,high[axis]+5,5):\n            segment=np.array([low-5,high+5]);segment[:,axis]=value\n            points=project(segment,inverse)\n            if np.isfinite(points).all() and abs(points).max()<100000:\n                cv2.line(overlay,tuple(points[0].astype(int)),tuple(points[1].astype(int)),(0,220,255),2)\n    selected=(mask>0)&np.any(overlay>0,axis=2);frame[selected]=overlay[selected]\n    cv2.polylines(frame,[hull.astype(int)],True,(255,180,0),2)\n    return frame\n\n\ndef render_pair(video,folder,observations,report,tracks,result,reference_kmh):\n    folder.mkdir(parents=True,exist_ok=True)\n    source=cv2.VideoCapture(str(video));assert source.isOpened()\n    fps=float(source.get(cv2.CAP_PROP_FPS));width=int(source.get(3));height=int(source.get(4))\n    size=(1280,720);scale=np.array([1280/width,720/height])\n    by_frame={};box_frames={}\n    for row in observations:by_frame.setdefault(row[\'frame_index\'],[]).append(row)\n    for row in tracks:box_frames.setdefault(row[\'frame_index\'],[]).append(row)\n    raw={name:folder/(name+\'.avi\') for name in [\'calibration\',\'speed\']}\n    writers={name:cv2.VideoWriter(str(path),cv2.VideoWriter_fourcc(*\'MJPG\'),fps,size) for name,path in raw.items()}\n    assert all(writer.isOpened() for writer in writers.values())\n    retained=target_records(observations,report,80,500)\n    hull=result.get(\'support_hull_image\',[])\n    H=np.array(result[\'mapping\']) if result.get(\'available\') else None\n    frames=0;labelled=0\n    estimate=result.get(\'estimated_kmh\');interval=result.get(\'time_interval_s\',[0,0])\n    title=f\'Estimated pass speed: {estimate:.1f} km/h\' if estimate is not None else \'Speed unavailable: insufficient calibration geometry\'\n    try:\n        while True:\n            ok,frame=source.read()\n            if not ok:break\n            calibration_frame=frame.copy()\n            if H is not None:calibration_frame=grid(calibration_frame,H,hull)\n            for row in by_frame.get(frames,[]):\n                points=np.array(row[\'keypoints\'])[report[\'ground_indices\'],:2].astype(int)\n                cv2.polylines(calibration_frame,[points],True,(0,255,0),3)\n                for node,point in enumerate(points):\n                    cv2.circle(calibration_frame,tuple(point),6,(0,255,255),-1)\n                    cv2.putText(calibration_frame,str(node),tuple(point+[8,-8]),cv2.FONT_HERSHEY_SIMPLEX,.6,(0,255,255),2)\n            calibration_frame=cv2.resize(calibration_frame,size);speed_frame=cv2.resize(frame,size)\n            timestamp=frames/fps\n            for row in box_frames.get(frames,[]):\n                p=np.array(row[\'box\']).reshape(2,2)*scale\n                active=result.get(\'available\') and row[\'track_id\']==result.get(\'track_id\') and interval[0]<=timestamp<=interval[1]\n                color=(0,255,255) if active else (160,160,160)\n                cv2.rectangle(speed_frame,tuple(p[0].astype(int)),tuple(p[1].astype(int)),color,2)\n                if active:\n                    labelled+=1;text=f\'{estimate:.1f} km/h\'\n                    pos=(max(5,int(p[0,0])),max(100,int(p[0,1])-8))\n                    cv2.putText(speed_frame,text,pos,cv2.FONT_HERSHEY_SIMPLEX,.85,(0,0,0),4)\n                    cv2.putText(speed_frame,text,pos,cv2.FONT_HERSHEY_SIMPLEX,.85,color,2)\n            for image in [calibration_frame,speed_frame]:\n                cv2.rectangle(image,(0,0),(1280,76),(20,20,20),-1)\n                cv2.rectangle(image,(0,677),(1280,720),(20,20,20),-1)\n            cv2.putText(calibration_frame,\'CALIBRATION REVIEW | ground nodes + local 5 m grid\',(15,29),cv2.FONT_HERSHEY_SIMPLEX,.7,(0,220,255),2)\n            method_label={\'reference_assisted_registered_road\':\'Saved calibration from reference passes\',\n                          \'automatic_single_pass_geometry\':\'New calibration from vehicle geometry\'}.get(result.get(\'method\'),\'Calibration unavailable\')\n            cv2.putText(calibration_frame,method_label,(15,58),cv2.FONT_HERSHEY_SIMPLEX,.53,(230,230,230),1)\n            cv2.putText(calibration_frame,\'Vehicle geometry is predicted. This is a local mapping, not a measured survey.\',(15,704),cv2.FONT_HERSHEY_SIMPLEX,.55,(230,230,230),1)\n            cv2.putText(speed_frame,title,(15,31),cv2.FONT_HERSHEY_SIMPLEX,.82,(0,255,255),2)\n            cv2.putText(speed_frame,f\'Average over source t={interval[0]:.2f}-{interval[1]:.2f} s | Experimental estimate\',(15,60),cv2.FONT_HERSHEY_SIMPLEX,.57,(230,230,230),1)\n            evaluation=f\'Cruise-control reference: {reference_kmh:.1f} km/h\'\n            if estimate is not None:evaluation+=f\' | Signed error: {estimate-reference_kmh:+.1f} km/h\'\n            cv2.putText(speed_frame,evaluation,(15,704),cv2.FONT_HERSHEY_SIMPLEX,.62,(230,230,230),1)\n            writers[\'calibration\'].write(calibration_frame);writers[\'speed\'].write(speed_frame)\n            if frames==round(sum(interval)/2*fps):cv2.imwrite(str(folder/\'speed_example.jpg\'),speed_frame)\n            frames+=1\n    finally:\n        source.release()\n        for writer in writers.values():writer.release()\n    for name,path in raw.items():\n        subprocess.run([\'ffmpeg\',\'-y\',\'-loglevel\',\'error\',\'-i\',str(path),\'-c:v\',\'libx264\',\'-crf\',\'25\',\n                        \'-pix_fmt\',\'yuv420p\',\'-movflags\',\'+faststart\',str(folder/(name+\'.mp4\'))],check=True)\n        path.unlink()\n    (folder/\'render_report.json\').write_text(json.dumps({\'frames\':frames,\'fps\':fps,\'duration_s\':frames/fps,\n        \'speed_display_decimal_places\':1,\'vehicle_label_frames\':labelled,\n        \'files\':[\'calibration.mp4\',\'speed.mp4\']},indent=2),encoding=\'utf-8\')\n\n\ndef run(args):\n    source=Path(args.inference_dir);videos=Path(args.video_dir);out=Path(args.output)\n    out.mkdir(parents=True,exist_ok=True)\n    calibration=json.loads(Path(args.calibration).read_text())\n    reference=cv2.imread(str(source/\'MercedesAMG550_30/reference.jpg\'));assert reference is not None\n    rows=[];predictions={}\n    # Freeze all configuration before any held-out prediction is compared with its label.\n    frozen={\'pilot_videos\':sorted(PILOT_NAMES),\'reference_calibration\':calibration,\n            \'automatic_config\':AUTOMATIC_CONFIG,\'legacy_config\':LEGACY_CONFIG,\n            \'prediction_source_sha256\':file_sha256(ROOT/\'validation_experiment.py\'),\n            \'reference_calibration_source_sha256\':file_sha256(ROOT/\'reference_calibration.py\'),\n            \'geometry_source_sha256\':file_sha256(ROOT/\'calibrate_speed.py\'),\n            \'benchmark_source_sha256\':file_sha256(ROOT/\'benchmark_speed.py\'),\n            \'reference_type\':\'User-confirmed cruise control setting from filename; .txt files ignored\',\n            \'network_fine_tuned\':False}\n    (out/\'frozen_calibration.json\').write_text(json.dumps(frozen,indent=2),encoding=\'utf-8\')\n    for folder in sorted(source.iterdir()):\n        if not (folder/\'observations.json\').exists():continue\n        video_name=folder.name+\'.MP4\';obs,report=load_clip(folder)\n        image=cv2.imread(str(folder/\'reference.jpg\'))\n        initial=estimate_single_pass(obs,report,LEGACY_CONFIG)\n        automatic=estimate_single_pass(obs,report,AUTOMATIC_CONFIG)\n        final=predict(obs,report,image,reference,calibration)\n        predictions[video_name]=final\n        # The reference enters only here, after predictions have been produced.\n        reference_kmh=float(folder.name.rsplit(\'_\',1)[1])\n        row={\'video\':video_name,\'split\':\'pilot\' if video_name in PILOT_NAMES else \'holdout\',\n             \'reference_kmh\':reference_kmh,\'legacy_diagnostic_kmh\':initial.get(\'estimated_kmh\'),\n             \'corrected_automatic_kmh\':automatic.get(\'estimated_kmh\'),\'estimated_kmh\':final.get(\'estimated_kmh\'),\n             \'method\':final.get(\'method\'),\'available\':final.get(\'available\',False),\n             \'calibration_reused\':final.get(\'calibration_reused\',False),\n             \'calibration_lookup_reason\':final.get(\'calibration_lookup_reason\'),\n             \'reference_used_during_calibration\':video_name in PILOT_NAMES,\n             \'trajectory_rms_m\':final.get(\'trajectory_rms_m\')}\n        estimate=row[\'estimated_kmh\'];error=None if estimate is None else estimate-reference_kmh\n        row.update(signed_error_kmh=error,absolute_error_kmh=None if error is None else abs(error),\n                   relative_error_percent=None if error is None else abs(error)/reference_kmh*100,\n                   within_2_kmh=False if error is None else abs(error)<=2)\n        rows.append(row)\n        archive=out/\'inference\'/folder.name;archive.mkdir(parents=True,exist_ok=True)\n        for name in [\'observations.json\',\'geometry_run_report.json\',\'tracks.json\',\'reference.jpg\']:\n            shutil.copy2(folder/name,archive/name)\n        if not args.no_render:\n            render_pair(videos/video_name,out/\'videos\'/folder.name,obs,report,\n                        json.loads((folder/\'tracks.json\').read_text()),final,reference_kmh)\n        print(video_name,\'reference\',reference_kmh,\'estimate\',None if estimate is None else round(estimate,2),\n              \'error\',None if error is None else round(error,2),row[\'method\'],flush=True)\n    with (out/\'results.csv\').open(\'w\',newline=\'\',encoding=\'utf-8\') as stream:\n        writer=csv.DictWriter(stream,fieldnames=list(rows[0]));writer.writeheader();writer.writerows(rows)\n    (out/\'results.json\').write_text(json.dumps(rows,indent=2),encoding=\'utf-8\')\n    (out/\'predictions.json\').write_text(json.dumps(predictions,indent=2),encoding=\'utf-8\')\n    metrics={split:{column:statistics([r for r in rows if split==\'all\' or r[\'split\']==split],column)\n                    for column in [\'legacy_diagnostic_kmh\',\'corrected_automatic_kmh\',\'estimated_kmh\']}\n             for split in [\'pilot\',\'holdout\',\'all\']}\n    metrics[\'holdout_by_method\']={method:statistics([r for r in rows if r[\'split\']==\'holdout\' and r[\'method\']==method],\'estimated_kmh\')\n        for method in sorted({r[\'method\'] for r in rows})}\n    (out/\'metrics.json\').write_text(json.dumps(metrics,indent=2),encoding=\'utf-8\')\n    make_report(out,rows,metrics)\n\n\ndef make_report(out,rows,metrics):\n    holdout=metrics[\'holdout\'][\'estimated_kmh\'];pilot=metrics[\'pilot\'][\'estimated_kmh\']\n    def number(value):return \'unavailable\' if value is None else f\'{value:.2f}\'\n    lines=[\'# Mercedes speed validation report\',\'\',\n        \'Run date: 2026-10-16 (Asia/Karachi). References are the user-confirmed cruise-control settings in filenames. Text annotation files are ignored.\',\n        \'\',f"**Held-out result: MAE {number(holdout.get(\'mae_kmh\'))} km/h; {holdout.get(\'within_2_kmh\',0)}/{holdout[\'clips\']} clips within ±2 km/h; {holdout[\'estimated\']}/{holdout[\'clips\']} estimated.**",\'\',\n        \'## Protocol\',\'\',\n        \'The first development clips were 30, 60 and 100 km/h. They were used to diagnose geometry and fit the reference-assisted calibration. The remaining 27 speeds were reserved for evaluation; their reference speeds are read only after each prediction is produced. Calibration, model hashes and configuration are saved in [frozen_calibration.json](frozen_calibration.json).\',\n        \'\', \'Pilot fitting results are not independent accuracy measurements. The held-out set still shares the same car and recording location, so it does not establish performance on other cars or the original CCTV camera.\',\n        \'\', \'## What changed\',\'\',\n        \'1. The video now displays a single point estimate with one decimal place. Sensitivity ranges remain in the original CCTV data files.\',\n        \'2. Separate calibration-review and speed-estimation videos are produced for each evaluated clip.\',\n        \'3. Ground-corner ordering now follows the checkpoint topology instead of selecting length edges from image motion; that heuristic frequently selected width edges in the roadside view.\',\n        \'4. Projective normalization is centred on observed footprints and the solver permits low-camera perspectives with a horizon near image centre. A synthetic regression test covers that case.\',\n        \'5. Reference-assisted calibration uses only the three pilot speeds and constant-motion trajectories. It is transferred only through verified road-texture registration. Unmatched views use an explicitly labelled automatic single-pass geometry estimate.\',\n        \'\', \'No neural-network weights were fine-tuned. These are calibration and estimation changes. The single-pass fallback uses an assumed 4.9 m × 1.87 m vehicle prior; exact vehicle dimensions were not independently confirmed.\',\n        \'\', \'## Before and after\',\'\',\n        \'The original production workflow cannot calibrate a clip containing only one car track: it requires three tracks. Legacy numbers below are explicitly **single-pass diagnostic estimates** made by relaxing that eligibility rule, not outputs that passed the old production checks.\',\n        \'\', \'| Split | Legacy diagnostic MAE | Corrected geometry MAE | Final MAE | Final within ±2 km/h |\',\n        \'|---|---:|---:|---:|---:|\']\n    for split in [\'pilot\',\'holdout\',\'all\']:\n        m=metrics[split];f=m[\'estimated_kmh\']\n        lines.append(f"| {split} | {number(m[\'legacy_diagnostic_kmh\'].get(\'mae_kmh\'))} | {number(m[\'corrected_automatic_kmh\'].get(\'mae_kmh\'))} | {number(f.get(\'mae_kmh\'))} | {f.get(\'within_2_kmh\',0)}/{f[\'clips\']} |")\n    lines+=[\'\',\'![Calibration changes: pilot and held-out MAE](calibration_progress.jpg)\',\n            \'\', \'![Point estimates and all errors](validation_errors.jpg)\',\n            \'\', \'## Held-out results by calibration path\',\'\',\n            \'| Calibration path | Clips | MAE km/h | Maximum error | Within ±2 |\',\n            \'|---|---:|---:|---:|---:|\']\n    for method,m in metrics.get(\'holdout_by_method\',{}).items():\n        lines.append(f"| {method} | {m[\'clips\']} | {number(m.get(\'mae_kmh\'))} | {number(m.get(\'maximum_absolute_error_kmh\'))} | {m.get(\'within_2_kmh\',0)}/{m[\'clips\']} |")\n    lines+=[\'\',\'## Every clip\',\'\',\n            \'| Clip / videos | Split | Reference | Initial diagnostic | Corrected geometry | Final estimate | Signed error | Within ±2 |\',\n            \'|---|---|---:|---:|---:|---:|---:|---|\']\n    for row in sorted(rows,key=lambda r:r[\'reference_kmh\']):\n        stem=Path(row[\'video\']).stem\n        link=f"[{row[\'video\']}](videos/{stem}/speed.mp4) · [calibration](videos/{stem}/calibration.mp4)"\n        lines.append(f"| {link} | {row[\'split\']} | {row[\'reference_kmh\']:.1f} | {number(row[\'legacy_diagnostic_kmh\'])} | {number(row[\'corrected_automatic_kmh\'])} | {number(row[\'estimated_kmh\'])} | {number(row[\'signed_error_kmh\'])} | {\'Yes\' if row[\'within_2_kmh\'] else \'No\'} |")\n    lines+=[\'\',\'## Error analysis\',\'\',\n        f"Held-out RMSE: {number(holdout.get(\'rmse_kmh\'))} km/h. Maximum absolute error: {number(holdout.get(\'maximum_absolute_error_kmh\'))} km/h. Signed bias: {number(holdout.get(\'bias_kmh\'))} km/h. Errors above 10 km/h: {holdout.get(\'errors_over_10_kmh\',0)}.",\n        \'\', \'All available point estimates are included in these metrics, including poor results. Missing estimates count against coverage and against the within-tolerance total. No clip is removed because its reference error is large.\',\n        \'\', \'**The requested accuracy target is met only if the held-out errors support it.** Cruise control supplies a reference setting, not an independently measured radar speed. Camera perspective, lens distortion, incorrect hidden ground corners and uncertain vehicle dimensions can still cause substantial errors. A learned cuboid is not a measured ground survey.\',\n        \'\', \'## Artifacts and reproduction\',\'\',\n        \'- [Detailed results CSV](results.csv), [metrics JSON](metrics.json), [prediction diagnostics](predictions.json).\',\n        \'- [Frozen calibration and provenance](frozen_calibration.json); [saved speed-blind inference inputs](inference/).\',\n        \'- All clips have a calibration video and a speed video under `videos/`. Speed videos show the cruise reference only as an evaluation overlay; it is not provided to the prediction function.\',\n        \'- Reproduce with `python validation_experiment.py --inference-dir output/mercedes_validation/inference --video-dir <original-video-folder> --calibration output/mercedes_validation/frozen_calibration.json --output outputs/validation_reproduction` (use the `reference_calibration` object when loading a frozen bundle).\',\n        \'\', \'## Research context\',\'\',\n        \'The filenames, speed list and scenes match the Mercedes subset of the [VS13 dataset](https://slobodan.ucg.ac.me/science/vs13/), whose authors describe cruise-control recordings from varied camera positions. The experiment uses the owner-provided clips and filename references. The approach uses the [UrbanOmniDetect checkpoint](https://github.com/mkturkcan/urbanomnidetect) and is inspired by [Sochor et al.](https://arxiv.org/abs/1702.06451).\',\n        \'\', \'## Conclusion\',\'\']\n    if holdout.get(\'maximum_absolute_error_kmh\',float(\'inf\'))<=2 and holdout[\'estimated\']==holdout[\'clips\']:\n        lines.append(\'All held-out clips are within the requested ±2 km/h tolerance relative to the cruise-control references. Further tests across independent vehicles, camera sites and measured reference speeds are still required.\')\n    else:\n        lines.append(\'The requested ±1–2 km/h accuracy has not been demonstrated across this held-out set. The report preserves failures as well as improvements. Further calibration work is required before treating these estimates as reliable: measured scene distances or independently verified vehicle geometry, lens calibration, and reference passes for unmatched camera setups would reduce the current ambiguity. This benchmark calibration must not be copied to the original CCTV camera.\')\n    (out/\'README.md\').write_text(\'\\n\'.join(lines)+\'\\n\',encoding=\'utf-8\')\n    available=[r for r in rows if r[\'estimated_kmh\'] is not None]\n    fig,axes=plt.subplots(1,2,figsize=(13,5))\n    for split,color in [(\'pilot\',\'orange\'),(\'holdout\',\'steelblue\')]:\n        subset=[r for r in available if r[\'split\']==split]\n        axes[0].scatter([r[\'reference_kmh\'] for r in subset],[r[\'estimated_kmh\'] for r in subset],label=split,c=color)\n        axes[1].scatter([r[\'reference_kmh\'] for r in subset],[r[\'signed_error_kmh\'] for r in subset],label=split,c=color)\n    axes[0].plot([25,110],[25,110],\'k--\');axes[0].set(xlabel=\'Cruise reference km/h\',ylabel=\'Estimated km/h\',title=\'Point estimates against reference\')\n    axes[1].axhspan(-2,2,color=\'green\',alpha=.15,label=\'±2 km/h\');axes[1].axhline(0,color=\'black\',lw=.5)\n    axes[1].set(xlabel=\'Cruise reference km/h\',ylabel=\'Signed error km/h\',title=\'All errors retained\')\n    for axis in axes:axis.grid(alpha=.2);axis.legend()\n    fig.tight_layout();fig.savefig(out/\'validation_errors.jpg\',dpi=150);plt.close(fig)\n    fig,axis=plt.subplots(figsize=(10,5))\n    columns=[\'legacy_diagnostic_kmh\',\'corrected_automatic_kmh\',\'estimated_kmh\']\n    labels=[\'Legacy diagnostic\',\'Corrected geometry\',\'Reference calibration + fallback\']\n    locations=np.arange(2)\n    for index,(column,label) in enumerate(zip(columns,labels)):\n        values=[metrics[split][column].get(\'mae_kmh\',0) for split in [\'pilot\',\'holdout\']]\n        axis.bar(locations+(index-1)*.25,values,.25,label=label)\n    axis.set_xticks(locations,[\'Pilot: calibration data\',\'Held-out: evaluation only\'])\n    axis.set_ylabel(\'Mean absolute error, km/h\');axis.set_title(\'Before/after calibration — fitting and evaluation kept separate\')\n    axis.legend();axis.grid(axis=\'y\',alpha=.2);fig.tight_layout();fig.savefig(out/\'calibration_progress.jpg\',dpi=150);plt.close(fig)\n\n\nif __name__==\'__main__\':\n    parser=argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--inference-dir\',required=True);parser.add_argument(\'--video-dir\',required=True)\n    parser.add_argument(\'--calibration\',required=True);parser.add_argument(\'--output\',required=True)\n    parser.add_argument(\'--no-render\',action=\'store_true\')\n    args=parser.parse_args()\n    # A frozen bundle can be supplied directly for reproduction.\n    bundle=json.loads(Path(args.calibration).read_text())\n    if \'reference_calibration\' in bundle:\n        temporary=Path(args.output)/\'reference_calibration_reproduction.json\';temporary.parent.mkdir(parents=True,exist_ok=True)\n        temporary.write_text(json.dumps(bundle[\'reference_calibration\']),encoding=\'utf-8\');args.calibration=str(temporary)\n    run(args)\n'}
calibration={'pilot_videos': ['MercedesAMG550_100.MP4', 'MercedesAMG550_30.MP4', 'MercedesAMG550_60.MP4'], 'reference_calibration': {'H_image_to_meters': [[0.15994911535045542, -0.15379978602984726, -9.03553865297072], [0.0, 0.06067165053214372, -39.1178495991835], [0.0041926824051557185, 0.001035571095912451, -2.5038177586186996]], 'support_hull_image': [[1640.62744140625, 904.2709350585938], [1483.3035888671875, 932.2748413085938], [1286.0745849609375, 934.996337890625], [1134.9891357421875, 910.98876953125], [953.2894897460938, 834.5346069335938], [600.8797607421875, 620.6361694335938], [590.8878173828125, 611.5972290039062], [589.0911865234375, 608.4996948242188], [588.5013427734375, 607.4559326171875], [635.3618774414062, 611.56005859375]], 'fit_rms': 0.9422698816858058, 'parameters': [4.02497510894949, 0.9941482520759529, 0.9693791889290392, -2.534953057661822, 4.046250929613946], 'reference_videos': ['MercedesAMG550_30.MP4', 'MercedesAMG550_60.MP4', 'MercedesAMG550_100.MP4'], 'reference_speeds_kmh': [30, 60, 100], 'method': 'reference_assisted_local_projective_calibration', 'physical_accuracy_verified': False, 'scope': 'Registered road region; this vehicle and recording setup.', 'metric_scale_refinement': {'multiplier': 1.0185740128514322, 'objective': 'minimum worst absolute pilot speed error', 'maximum_pilot_error_kmh': 1.863144306031199, 'unrefined_pilot_estimates_kmh': [29.998381890185243, 57.076712109725264, 100.00563829512191]}}, 'frozen_before_heldout_evaluation': True, 'reference_label_files_used': False, 'neural_network_fine_tuned': False, 'source_hashes': {'calibrate_speed.py': '52711385c2fe4532b92791d303669175ed8d03be6a772b1d25ae25729eff93c9', 'reference_calibration.py': 'f8b8c2a1776de48e9e8c8f1a2a6bf34bf7b2d85799ff78621168122f1818645c', 'benchmark_speed.py': '3bf9b959dd66a3eb1404a35f9fc5207a78760f83b8046848e987380b8752d93a'}}
root=Path('/kaggle/working/pipeline');root.mkdir(exist_ok=True)
for name,source in sources.items():(root/name).write_text(source,encoding='utf-8')
frozen=Path('/kaggle/working/frozen_pilot_calibration.json')
frozen.write_text(json.dumps(calibration),encoding='utf-8')
input_root=Path('/kaggle/input')
matches=[p.parent.parent for p in input_root.rglob('observations.json') if p.parent.name=='MercedesAMG550_30']
assert len(matches)==1,matches
inference=matches[0]
assert len(list(inference.glob('*/observations.json')))==30,'Wait for the full 30-clip GPU run to finish.'
videos=list(input_root.rglob('MercedesAMG550_30.MP4'));assert len(videos)==1,videos
subprocess.check_call([sys.executable,str(root/'validation_experiment.py'),
    '--inference-dir',str(inference),'--video-dir',str(videos[0].parent),
    '--calibration',str(frozen),'--output','/kaggle/working/mercedes_validation'])


In [ ]:
from IPython.display import display,Image,Video,Markdown
import pandas as pd
out=Path('/kaggle/working/mercedes_validation')
display(Markdown((out/'README.md').read_text()))
display(pd.read_csv(out/'results.csv'))
display(Image(filename=str(out/'validation_errors.jpg'),width=1100))
for speed in [30,60,80,100]:
    for name in ['calibration.mp4','speed.mp4']:
        display(Video(str(out/'videos'/f'MercedesAMG550_{speed}'/name),embed=True,width=1100))
